# RAG (Retrieval-Augmented Generation) with LangChain + HuggingFace + Pinecone

## What is a RAG?
A **Retrieval-Augmented Generation (RAG)** system is an AI architecture that combines two things:
1. A **retrieval system** that searches for relevant documents from a knowledge base
2. A **language model (LLM)** that generates answers using the retrieved documents as context

This approach improves the quality of responses by grounding them in real documents, reducing hallucinations.

## Architecture of this project
```
User Question
     ↓
[Embeddings Model] → converts question to vector
     ↓
[Pinecone Vector DB] → finds similar document chunks
     ↓
[Retrieved Context + Question] → sent to LLM
     ↓
[LLM] → generates final answer
     ↓
Answer
```

## Why HuggingFace instead of OpenAI?
The original lab specification requests OpenAI for both embeddings and the LLM. However, **OpenAI's API requires a paid account with credits** (minimum ~$5 USD). 

To make this lab fully runnable without any cost, we use:
- **`sentence-transformers/all-MiniLM-L6-v2`** for embeddings — a free, open-source model that runs locally
- **`google/flan-t5-base`** as the LLM — a free, open-source model from Google that runs locally

The RAG architecture, concepts, and LangChain implementation are **identical** to what you would build with OpenAI. Only the model provider changes.

## Step 1: Install Dependencies

We install all required libraries:
- `langchain` — the main framework for building RAG pipelines
- `langchain-pinecone` — LangChain integration with Pinecone vector database
- `langchain-huggingface` — LangChain integration with HuggingFace models
- `langchain-community` — community document loaders and utilities
- `pinecone-client` — to connect to Pinecone
- `sentence-transformers` — for the free local embedding model
- `transformers` and `torch` — to run HuggingFace models locally
- `python-dotenv` — to securely load API keys from a `.env` file

In [ ]:
!pip install langchain langchain-pinecone langchain-huggingface langchain-community pinecone-client sentence-transformers python-dotenv transformers torch

## Step 2: Configure your `.env` file

Create a file named `.env` in the same folder as this notebook with the following content:

```
PINECONE_API_KEY=your_pinecone_key_here
PINECONE_INDEX_NAME=rag-demo
```

**Where to get your Pinecone API key:**
1. Go to [app.pinecone.io](https://app.pinecone.io) and create a free account
2. Navigate to **API Keys** in the left menu
3. Copy your key and paste it in the `.env` file

> ⚠️ Never write your API keys directly in the notebook. Always use a `.env` file.

## Step 3: Import Libraries and Load Environment Variables

In [ ]:
import os
from dotenv import load_dotenv

from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_pinecone import PineconeVectorStore
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
from pinecone import Pinecone, ServerlessSpec

# Load environment variables from .env file
load_dotenv(".env", override=True)

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
PINECONE_INDEX_NAME = os.getenv("PINECONE_INDEX_NAME", "rag-demo")

# Verify keys are loaded correctly (shows only first 10 chars for security)
print("Pinecone key loaded:", PINECONE_API_KEY[:10] + "..." if PINECONE_API_KEY else "NOT FOUND")
print("Index name:", PINECONE_INDEX_NAME)

## Step 4: Create a Sample Document

We create a simple text file that will serve as our knowledge base.
In a real project, this could be PDFs, web pages, documentation, etc.

The RAG system will retrieve information from this document to answer questions.

In [ ]:
os.makedirs("data", exist_ok=True)

sample_text = """
Vector databases store embeddings as high-dimensional vectors.
They enable similarity search using cosine distance.
Popular vector databases include Pinecone, Weaviate, and ChromaDB.
Pinecone is a managed vector database that offers a free tier for small projects.

Retrieval-Augmented Generation (RAG) combines retrieval systems with large language models.
This architecture improves factual grounding and reduces hallucinations.
RAG systems first retrieve relevant documents, then use them as context for generation.
The main advantage of RAG is that it allows LLMs to access external knowledge without retraining.

Embeddings convert text into numerical representations that preserve semantic meaning.
Similar texts produce embeddings that are close together in vector space.
The all-MiniLM-L6-v2 model from sentence-transformers produces embeddings of 384 dimensions.

LangChain is a framework for building applications with large language models.
It provides tools for chaining components like retrievers, prompts, and models.
LangChain supports many vector stores and embedding providers including Pinecone and HuggingFace.
"""

with open("data/sample.txt", "w", encoding="utf-8") as f:
    f.write(sample_text)

print("Sample document created at: data/sample.txt")

## Step 5: Load and Split the Document into Chunks

We cannot store an entire document as a single vector — it would lose too much detail.
Instead, we split it into smaller **chunks** that are then individually embedded.

- `chunk_size=500` — each chunk has at most 500 characters
- `chunk_overlap=50` — chunks overlap by 50 characters to preserve context at boundaries

In [ ]:
loader = TextLoader("data/sample.txt", encoding="utf-8")
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

docs = text_splitter.split_documents(documents)

print(f"Total chunks created: {len(docs)}")
for i, doc in enumerate(docs):
    print(f"\nChunk {i+1} ({len(doc.page_content)} chars):")
    print(doc.page_content[:120] + "...")

## Step 6: Load the Embedding Model

Embeddings transform text into numerical vectors so we can measure semantic similarity.

We use **`sentence-transformers/all-MiniLM-L6-v2`** which:
- Is **completely free** and runs locally on your machine
- Produces vectors of **384 dimensions**
- Is fast and works well for semantic search tasks

> **Note:** If we were using OpenAI (as the lab originally specifies), we would use
> `OpenAIEmbeddings(model="text-embedding-ada-002")` which produces 1536-dimension vectors
> but requires a paid API key.

In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully (384 dimensions, free, runs locally)")

## Step 7: Create Pinecone Index and Store Document Embeddings

Pinecone is our **vector database** — it stores the embeddings and allows fast similarity search.

Important: the index `dimension` must match the embedding model output:
- `all-MiniLM-L6-v2` → **384 dimensions**
- OpenAI `text-embedding-ada-002` → 1536 dimensions

If you already have an index with a different dimension, you must delete it first and recreate it.

In [ ]:
pc = Pinecone(api_key=PINECONE_API_KEY)

existing_indexes = pc.list_indexes().names()
print("Existing indexes:", existing_indexes)

# Delete index if it exists with wrong dimension
if PINECONE_INDEX_NAME in existing_indexes:
    index_info = pc.describe_index(PINECONE_INDEX_NAME)
    if index_info.dimension != 384:
        print(f"Index exists with wrong dimension ({index_info.dimension}). Deleting and recreating...")
        pc.delete_index(PINECONE_INDEX_NAME)
        existing_indexes = []

if PINECONE_INDEX_NAME not in existing_indexes:
    print(f"Creating index '{PINECONE_INDEX_NAME}' with 384 dimensions...")
    pc.create_index(
        name=PINECONE_INDEX_NAME,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )
    print("Index created successfully")
else:
    print(f"Index '{PINECONE_INDEX_NAME}' already exists with correct dimension")

# Store document embeddings in Pinecone
vectorstore = PineconeVectorStore.from_documents(
    docs,
    embeddings,
    index_name=PINECONE_INDEX_NAME
)

print("\nDocuments successfully indexed in Pinecone!")

## Step 8: Configure the Retriever

The **retriever** is the component that searches Pinecone for the most relevant document chunks
given a user question. It converts the question to a vector and finds the closest matches.

`k=3` means it will return the **3 most similar chunks** for each query.

In [ ]:
vectorstore = PineconeVectorStore(
    index_name=PINECONE_INDEX_NAME,
    embedding=embeddings
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print("Retriever configured — will return top 3 most relevant chunks per query")

## Step 9: Load the Language Model (LLM)

The LLM reads the retrieved context and generates a human-readable answer.

We use **`google/flan-t5-base`** which:
- Is **completely free** and runs locally on your machine
- Is a instruction-following model from Google, better suited for Q&A than GPT-2
- Works well for reading a context and answering questions about it

> **Note:** If we were using OpenAI (as the lab originally specifies), we would use
> `ChatOpenAI(model="gpt-3.5-turbo")` which gives much better responses
> but requires a paid API key (~$5 minimum).
>
> We deliberately avoid `gpt2` here because despite having 'GPT' in its name,
> it is a very old (2019) model that generates poor, incoherent answers for Q&A tasks.

In [ ]:
llm = HuggingFacePipeline.from_model_id(
    model_id="google/flan-t5-base",
    task="text2text-generation",  # flan-t5 uses text2text, not text-generation like GPT-2
    pipeline_kwargs={
        "max_new_tokens": 200,
        "do_sample": False
    }
)

print("LLM loaded: google/flan-t5-base (free, runs locally)")

## Step 10: Build the RAG Pipeline

Now we connect all components into a single chain:

1. **Prompt** — defines how to combine the retrieved context with the user question
2. **`create_stuff_documents_chain`** — takes the retrieved docs and "stuffs" them into the prompt
3. **`create_retrieval_chain`** — connects the retriever to the document chain

The final `rag_chain` takes a question as input and returns a full answer.

In [ ]:
prompt = ChatPromptTemplate.from_template("""
Use the following context to answer the question at the end.
If you cannot find the answer in the context, say "I don't have enough information to answer that."

Context:
{context}

Question:
{input}

Answer:
""")

doc_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, doc_chain)

print("RAG pipeline built successfully!")
print("Flow: Question → Retriever → Pinecone → Context → LLM → Answer")

## Step 11: Query the RAG System

Now we test the RAG system with several questions.
For each question, the system will:
1. Search Pinecone for relevant chunks
2. Pass those chunks + the question to the LLM
3. Return a grounded answer

In [ ]:
questions = [
    "What is a vector database?",
    "What is RAG and what problem does it solve?",
    "What is LangChain used for?",
    "How many dimensions does the all-MiniLM-L6-v2 model produce?"
]

for question in questions:
    print(f"\n{'='*60}")
    print(f"Question: {question}")
    print("-"*60)
    response = rag_chain.invoke({"input": question})
    print(f"Answer: {response['answer']}")

## Step 12: Inspect Retrieved Documents

This step shows which document chunks the retriever found for a given query.
This is useful to understand **why** the LLM gave a certain answer — it can only answer
based on what was retrieved.

In [ ]:
query = "What is LangChain?"
retrieved_docs = retriever.invoke(query)

print(f"Retrieved chunks for: '{query}'\n")
for i, doc in enumerate(retrieved_docs):
    print(f"--- Chunk {i+1} ---")
    print(doc.page_content)
    print()

## Summary

In this notebook we built a complete RAG system using:

| Component | Tool Used | Why |
|---|---|---|
| Document Loader | LangChain TextLoader | Loads text files into LangChain documents |
| Text Splitter | RecursiveCharacterTextSplitter | Splits docs into manageable chunks |
| Embeddings | HuggingFace all-MiniLM-L6-v2 | Free, local, 384-dim semantic vectors |
| Vector Database | Pinecone | Stores and retrieves embeddings by similarity |
| LLM | Google flan-t5-base | Free, local, good at instruction following |
| Framework | LangChain | Connects all components into a pipeline |

> If OpenAI credits were available, `HuggingFaceEmbeddings` would be replaced by `OpenAIEmbeddings`
> and `HuggingFacePipeline` by `ChatOpenAI`, as originally specified in the lab.
> The rest of the architecture remains exactly the same.